# Prerequisites

## Load data

In [0]:
df = spark.read.csv(path='/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv',
                    header=True,
                    inferSchema=True)

# Transcation 01: Write into the delta format

In [0]:
df.write.format('delta').save('/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta',mode="overwri")

## Read delta

In [0]:
df_delta = spark.read.load(path  = '/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta')
df_delta.limit(4).display()

# Transcation 02

In [0]:
from pyspark.sql.functions import col
df.filter(col("country") == "India").write.format('delta').save('/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta', mode='OVERWRITE')

# Read the transaction log

In [0]:
spark.read.text("/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta/_delta_log/00000000000000000000.json").display()

# APPROACH 02

In [0]:
from delta.tables import DeltaTable
delta_table=DeltaTable.forPath(spark,"/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta/" )
delta_table.history().display()

In [0]:
df_delta.createOrReplaceTempView('user_vw')

# How to Overcome Drawbacks

## Update

In [0]:
%sql
update user_vw
set country='Bharath'
where country='India'

In [0]:
from pyspark.sql.functions import col
df_delta.filter(col('country')=='Bharath').display()

## Roll Back

In [0]:
df_delta.filter(col('country')=='Bharath').write.format('delta').save('/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta', mode='OVERWRITE')

df_delta2 = spark.read.load('/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta')
df_delta2.limit(4).display()

In [0]:
df_delta2.count()

In [0]:
spark.sql(" DESCRIBE HISTORY delta.`/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_delta`").display()